# Caso J · 04 Integración tráfico × meteorología — predicción congestión

> _Tutorial · Caso de uso: **J — Tráfico + YOLO** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Cruzar conteos de tráfico con meteorología (lluvia) y entrenar un modelo que prediga `congestion_level`.


## 2. Qué se aprende

- Merge time-aligned.
- Random Forest para clasificación ordinal.
- Feature importance climática.


## 3. Contexto del caso de uso

Capa oro: predicción para los próximos 15 min.


## 4. Relación con CENTINELA+

Tool del chatbot Caso H opcional.


## 5. Relación con Medallion

Lee plata, escribe oro.


## 6. Datos de entrada

Mock traffic + AEMET (incluido en mismo CSV).


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica (oro).


## 9. Carga de datos o mock

Cargamos.


In [2]:
csv_path = ROOT / "notebooks/_data/traffic_camera_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.dayofweek
df["is_rush"] = ((df["hour"].isin([7, 8, 9, 17, 18, 19])) & (df["weekday"] < 5)).astype(int)
df.head()


,timestamp,camera_id,vehicle_count,congestion_level,detection_confidence,precip_mm,hour,weekday,is_rush
0,2024-09-01 00:00:00+00:00,DGT_CAM_V46_001,36,1,0.869,0.08,0,6,0
1,2024-09-01 00:15:00+00:00,DGT_CAM_V46_001,29,1,0.876,0.08,0,6,0
2,2024-09-01 00:30:00+00:00,DGT_CAM_V46_001,30,1,0.958,0.08,0,6,0
3,2024-09-01 00:45:00+00:00,DGT_CAM_V46_001,38,1,0.841,0.00,0,6,0
4,2024-09-01 01:00:00+00:00,DGT_CAM_V46_001,36,1,0.818,0.00,1,6,0


## 10. Exploración paso a paso

Correlación.


In [3]:
print(df[["vehicle_count", "precip_mm", "is_rush", "congestion_level"]].corr().round(2))


                  vehicle_count  precip_mm  is_rush  congestion_level
vehicle_count              1.00       0.01     0.81              0.89
precip_mm                  0.01       1.00     0.01              0.01
is_rush                    0.81       0.01     1.00              0.72
congestion_level           0.89       0.01     0.72              1.00


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Modelo.


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = df[["vehicle_count", "precip_mm", "is_rush", "hour"]]
y = df["congestion_level"]
n = len(df); i = int(n * 0.7)
X_tr, X_te = X.iloc[:i], X.iloc[i:]
y_tr, y_te = y.iloc[:i], y.iloc[i:]

m = RandomForestClassifier(n_estimators=120, random_state=SEED).fit(X_tr, y_tr)
y_pred = m.predict(X_te)
print(classification_report(y_te, y_pred, zero_division=0))


              precision    recall  f1-score   support

           1       0.97      0.98      0.98       183
           2       0.97      0.97      0.97       155
           3       1.00      0.97      0.98        66

    accuracy                           0.98       404
   macro avg       0.98      0.97      0.98       404
weighted avg       0.98      0.98      0.98       404



## 13. Visualizaciones explicativas

Feature importance.


In [5]:
imp = pd.Series(m.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(color="#9C27B0", figsize=(7, 3))
plt.title("Feature importance — congestion_level")
plt.tight_layout()


## 14. Validaciones

El modelo discrimina niveles base.


In [6]:
from sklearn.metrics import balanced_accuracy_score
print({"balanced_acc": balanced_accuracy_score(y_te, y_pred)})


{'balanced_acc': 0.97368182085263}


## 15. Errores comunes

1. Predecir como regresión cuando es ordinal.
2. No incluir hora del día.
3. Comparar accuracy entre datasets desbalanceados.


## 16. Ejercicios propuestos

1. Añade `dvehicle_count_15min` como feature.
2. Reemplaza por `OrdinalRegression`.
3. Construye un dashboard Grafana con prediction live.


## 17. Cómo se reutiliza con datos reales

Cambiar el mock por la query Flux que combina `traffic_cameras` + `weather_station`.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Documento web del caso: `docs/use-cases/case-j-traffic-yolo.md`.
- ¡Has completado los 42 notebooks didácticos!
- Vuelve a `00_project_overview/00_arquitectura_medallion_captia.ipynb` para revisitar el mapa.


## 19. Marco teórico (nivel doctoral)

### YOLO v8 — single-stage anchor-free detector

Por cada celda de la grid, salida:

$$
\hat{y} = (b_x, b_y, b_w, b_h, p_{obj}, p_{c_1}, ..., p_{c_C})
$$

Loss combinada:

$$
\mathcal{L} = \lambda_{box} \mathcal{L}_{CIoU} + \lambda_{obj} \mathcal{L}_{BCE,obj} + \lambda_{cls} \mathcal{L}_{BCE,cls}
$$

### Series temporales tráfico

$$
N_v(t) = \sum_{i=1}^{D_t} \mathbb{1}[\text{detection}_i \in v_{ROI}]
$$

con NMS IoU threshold = 0.5.

### Predictor congestión

$$
\hat{C}(t+15) = \text{XGB}(N_v(t), N_v(t-15), ..., \text{weather}, t_{hora}, t_{dow})
$$

con $C \in \{0, 1, 2, 3\}$ niveles de congestión.

### Métricas

$$
\text{mAP}@0.5 = \frac{1}{|C|} \sum_{c \in C} \text{AP}_c \quad (\text{IoU} \geq 0.5)
$$

Objetivos: mAP@0.5 ≥ 0.90 (car/truck), ≥ 0.75 (motorbike/bicycle).


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Aunque tangencial al BMS de aulas, este caso demuestra que la **stack de IA + datos sintéticos + modelos** de CAPTIA es extensible a otros verticales (smart cities). Activo comercial para diversificar.

### ROI estimado

| Concepto | Valor |
|---|---|
| Predicción congestión 15 min (semáforos) | +5 000 €/año |
| Detección incidentes < 60 s (emergencias) | +12 000 €/año |
| **Bruto** | **+17 000 €/año** |
| Compute GPU dedicada | -1 500 €/año |
| **Neto** | **+15 500 €/año** |


## 21. Bibliografía y referencias

- Redmon, J. & Farhadi, A. (2018). *YOLOv3: An Incremental Improvement*. arXiv:1804.02767.
- Ultralytics (2024). *YOLOv8 Documentation*. https://docs.ultralytics.com
- Lin, T.-Y. et al. (2014). *Microsoft COCO: Common Objects in Context*. ECCV.
- DGT España. *Información en tiempo real*. http://infocar.dgt.es
